# CatVTON-FLUX Inference Server (Fine-Tuned)

Serves the fine-tuned CatVTON-FLUX LoRA model via Gradio API.
Downloads latest LoRA weights from Supabase, starts inference server.

In [ ]:
#@title 1. Setup
import os, subprocess

gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")

if not os.path.exists('/content/catvton-flux'):
    !git clone https://github.com/nftblackmagic/catvton-flux.git /content/catvton-flux

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate safetensors gradio requests

print("✅ Setup complete")

In [ ]:
#@title 2. Configuration
SUPABASE_URL = "https://qfumhgipfhzubmorymbd.supabase.co"  #@param {type:"string"}
SUPABASE_SERVICE_KEY = ""  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

In [ ]:
#@title 3. Download Latest LoRA Weights
import requests

lora_path = "/content/catvton-flux-lora.safetensors"
url = f"{SUPABASE_URL}/storage/v1/object/vto-images/models/catvton-flux-lora-latest.safetensors"

r = requests.get(url, headers={"Authorization": f"Bearer {SUPABASE_SERVICE_KEY}"})
if r.status_code == 200:
    with open(lora_path, 'wb') as f:
        f.write(r.content)
    print(f"✅ Downloaded LoRA weights: {len(r.content)/1024/1024:.1f}MB")
else:
    print(f"⚠️ No fine-tuned weights found ({r.status_code}). Using base CatVTON-FLUX LoRA.")
    lora_path = None

In [ ]:
#@title 4. Load Model & Start Gradio Server
import torch
import gradio as gr
from diffusers import FluxFillPipeline
from PIL import Image
import numpy as np
import base64, io, json, time

print("Loading FLUX.1-Fill pipeline...")
pipe = FluxFillPipeline.from_pretrained(
    "xiaozaa/flux1-fill-dev-diffusers",
    torch_dtype=torch.bfloat16
).to("cuda")

# Load LoRA weights
if lora_path and os.path.exists(lora_path):
    pipe.load_lora_weights(lora_path)
    print(f"✅ Fine-tuned LoRA loaded")
else:
    # Use base CatVTON-FLUX LoRA from HuggingFace
    pipe.load_lora_weights("xiaozaa/catvton-flux-lora-alpha", weight_name="catvton_flux_lora_alpha.safetensors")
    print(f"✅ Base CatVTON-FLUX LoRA loaded")

pipe.enable_model_cpu_offload()
print("✅ Model ready")

def tryon(person_url, garment_url, category="upper_body"):
    """Virtual try-on API endpoint."""
    start = time.time()
    try:
        # Download images
        person = Image.open(io.BytesIO(requests.get(person_url).content)).convert('RGB').resize((576, 768))
        garment = Image.open(io.BytesIO(requests.get(garment_url).content)).convert('RGB').resize((576, 768))
        
        # Generate upper body mask
        mask = np.zeros((768, 576), dtype=np.uint8)
        if category == 'upper_body':
            mask[int(768*0.15):int(768*0.60), int(576*0.10):int(576*0.90)] = 255
        elif category == 'lower_body':
            mask[int(768*0.50):int(768*0.90), int(576*0.10):int(576*0.90)] = 255
        else:
            mask[int(768*0.15):int(768*0.90), int(576*0.10):int(576*0.90)] = 255
        mask_img = Image.fromarray(mask, mode='L')
        
        # Concatenate cloth + masked person
        person_arr = np.array(person)
        person_arr[mask > 128] = 128
        concat = Image.new('RGB', (576*2, 768))
        concat.paste(garment, (0, 0))
        concat.paste(Image.fromarray(person_arr), (576, 0))
        
        # Generate
        result = pipe(
            image=concat,
            mask_image=mask_img,
            prompt="virtual try-on, photorealistic",
            height=768, width=576,
            num_inference_steps=30,
            guidance_scale=3.5,
        ).images[0]
        
        # Convert to base64
        buf = io.BytesIO()
        result.save(buf, format='JPEG', quality=90)
        b64 = base64.b64encode(buf.getvalue()).decode()
        
        return json.dumps({"success": True, "image_base64": b64, "duration_ms": int((time.time()-start)*1000)})
    except Exception as e:
        return json.dumps({"success": False, "error": str(e), "duration_ms": int((time.time()-start)*1000)})

# Launch Gradio
demo = gr.Interface(
    fn=tryon,
    inputs=[gr.Textbox(label="Person URL"), gr.Textbox(label="Garment URL"), gr.Textbox(label="Category", value="upper_body")],
    outputs=gr.Textbox(label="Result JSON"),
    title="CatVTON-FLUX Fine-Tuned VTO",
)

demo.launch(share=True)
print("\n🚀 Server running! Copy the public URL above.")